# Regional Indicator Gold

Stores regional development and service-access measurements for comparison with government spending.

**Target:** `pondolens.dev_gold.fact_regional_indicator`

## Grain and columns
- One row per `year_id + region_id + indicator_id`.
- Each row represents one specific indicator measurement for one region and year.
- `indicator_id` identifies the measure, service, unit, breakdown, and price basis.
- `measurement` comes from Silver.
- `unit`, `service`, and `breakdown_value` come from `dim_indicator`.
- `fact_indicator_id` is the surrogate key, starting at **50001**.

## Analytical coverage
- **BP:** Provides regional development context for assessing how government spending aligns with regional needs.
- **AP1:** Adds regional indicator context to spending by agency and region.
- **AP2:** Budget execution gaps are calculated using the regional expenditure fact.
- **AP3:** Supports comparisons of obligations with GRDP per capita, unemployment, and available water and internet access measurements.


## Loading rules
- Includes GRDP per capita at constant 2018 prices, unemployment rate, and service-access measurements.
- Reads service access from `pondolens.dev_silver.service_access_available`.
- Loads only non-null measurements with matching year, regional, and indicator dimension entries.
- Matches indicators using the measure type, unit, service, price basis, and breakdown.
- Rural and Urban breakdowns are not explicitly excluded.

In [0]:
%sql

CREATE OR REPLACE TABLE pondolens.dev_gold.fact_regional_indicator
USING DELTA
AS

WITH all_indicators AS (

    -- 1. GRDP PER CAPITA: 2018 PRICE BASIS ONLY
    SELECT
        CAST(year AS INT) AS observation_year,
        region_code,

        'grdp' AS indicator,
        'grdp_per_capita' AS measure_type,
        CAST(grdp_per_capita AS DECIMAL(20,8)) AS measurement,

        'pesos' AS unit,
        TRIM(CAST(price_basis AS STRING)) AS price_basis,
        'n/a' AS service,
        'total' AS breakdown_type,
        'all' AS breakdown_value

    FROM pondolens.dev_silver.grdp_per_capita

    WHERE LOWER(TRIM(geographic_level)) = 'region'
      AND TRIM(CAST(price_basis AS STRING)) = '2018'
      AND grdp_per_capita IS NOT NULL

    UNION ALL

    -- 2. UNEMPLOYMENT RATE
    SELECT
        CAST(year AS INT) AS observation_year,
        region_code,

        'unemployment' AS indicator,
        'unemployment_rate' AS measure_type,
        CAST(unemployment_rate AS DECIMAL(20,8)) AS measurement,

        '% of labor force' AS unit,
        'n/a' AS price_basis,
        'n/a' AS service,
        'total' AS breakdown_type,
        'all' AS breakdown_value

    FROM pondolens.dev_silver.unemployment_rate

    WHERE unemployment_rate IS NOT NULL

    UNION ALL

    -- 3. SERVICE ACCESS
    SELECT
        CAST(year AS INT) AS observation_year,
        region_code,

        'service_access' AS indicator,
        LOWER(REGEXP_REPLACE(TRIM(service_measure), '[ -]+', '_')) AS measure_type,
        CAST(value AS DECIMAL(20,8)) AS measurement,

        LOWER(TRIM(unit)) AS unit,
        'n/a' AS price_basis,
        LOWER(TRIM(service)) AS service,
        LOWER(TRIM(breakdown_type)) AS breakdown_type,
        LOWER(TRIM(breakdown_value)) AS breakdown_value

    FROM pondolens.dev_silver.service_access_available

    WHERE value IS NOT NULL
      AND COALESCE(UPPER(TRIM(region_code)), '') <> 'PHL'
      AND COALESCE(UPPER(TRIM(region)), '') <> 'PHILIPPINES'
),

mapped_indicators AS (

    SELECT
        y.year_id,
        r.region_id,
        d.indicator_id,
        s.indicator,
        s.measurement,
        d.unit,
        d.service,
        d.breakdown_value

    FROM all_indicators s

    INNER JOIN pondolens.dev_gold.dim_year y
        ON s.observation_year = y.year

    INNER JOIN pondolens.dev_gold.dim_location r
        ON REGEXP_REPLACE(
            UPPER(TRIM(s.region_code)),
            '[-\\s]',
            ''
        ) = REGEXP_REPLACE(
            UPPER(TRIM(r.region_code)),
            '[-\\s]',
            ''
        )
       AND UPPER(REGEXP_REPLACE(TRIM(r.geographic_level), '\\s+', ' ')) = 'ALL'

    INNER JOIN pondolens.dev_gold.dim_indicator d
        ON LOWER(TRIM(s.indicator)) = LOWER(TRIM(d.indicator))
       AND LOWER(TRIM(s.measure_type)) = LOWER(TRIM(d.measure_type))

       AND LOWER(TRIM(s.unit))
           <=> LOWER(TRIM(d.unit))

       AND LOWER(TRIM(s.price_basis))
           <=> LOWER(TRIM(d.price_basis))

       AND LOWER(TRIM(s.service))
           <=> LOWER(TRIM(d.service))

       AND LOWER(TRIM(s.breakdown_type))
           <=> LOWER(TRIM(d.breakdown_type))

       AND LOWER(TRIM(s.breakdown_value))
           <=> LOWER(TRIM(d.breakdown_value))
),

deduplicated AS (

    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY year_id, region_id, indicator_id
            ORDER BY measurement DESC
        ) AS rn

    FROM mapped_indicators
)

SELECT
    50000 + ROW_NUMBER() OVER (
        ORDER BY year_id, region_id, indicator_id
    ) AS fact_indicator_id,

    year_id,
    indicator_id,
    region_id,
    indicator,
    measurement,
    unit,
    service,
    breakdown_value

FROM deduplicated

WHERE rn = 1;


## Data Quality Check

In [0]:
%sql
SELECT *
FROM pondolens.dev_gold.fact_regional_indicator
ORDER BY year_id, region_id, indicator_id;

In [0]:
%sql
SELECT
    f.indicator,
    f.indicator_id,
    f.service,
    f.breakdown_value,
    f.unit,
    y.year,
    COUNT(DISTINCT f.region_id) AS total_regions,
    COUNT(*) AS total_records
FROM pondolens.dev_gold.fact_regional_indicator f
LEFT JOIN pondolens.dev_gold.dim_year y
    ON f.year_id = y.year_id
GROUP BY
    f.indicator,
    f.indicator_id,
    f.service,
    f.breakdown_value,
    f.unit,
    y.year
ORDER BY f.indicator, f.indicator_id, y.year;

In [0]:

%sql
WITH fact AS (
    SELECT
        *,
        COUNT(*) OVER (
            PARTITION BY fact_indicator_id
        ) AS id_count,
        COUNT(*) OVER (
            PARTITION BY year_id, region_id, indicator_id
        ) AS grain_count
    FROM pondolens.dev_gold.fact_regional_indicator
),

totals AS (
    SELECT COUNT(*) AS rows_inspected
    FROM fact
),

checks AS (
    SELECT
        1 AS check_order,
        'Missing fact or dimension IDs' AS dq_check,
        COUNT_IF(
            fact_indicator_id IS NULL
            OR year_id IS NULL
            OR region_id IS NULL
            OR indicator_id IS NULL
        ) AS affected_rows
    FROM fact

    UNION ALL

    SELECT
        2,
        'Duplicate fact IDs',
        COUNT_IF(id_count > 1)
    FROM fact

    UNION ALL

    SELECT
        3,
        'Duplicate year-region-indicator rows',
        COUNT_IF(grain_count > 1)
    FROM fact

    UNION ALL

    SELECT
        4,
        'Fact IDs below 50001',
        COUNT_IF(fact_indicator_id < 50001)
    FROM fact

    UNION ALL

    SELECT
        5,
        'Missing measurements or descriptive fields',
        COUNT_IF(
            measurement IS NULL
            OR NULLIF(TRIM(indicator), '') IS NULL
            OR NULLIF(TRIM(unit), '') IS NULL
            OR NULLIF(TRIM(service), '') IS NULL
            OR NULLIF(TRIM(breakdown_value), '') IS NULL
        )
    FROM fact

    UNION ALL

    SELECT
        6,
        'Year IDs missing from dim_year',
        COUNT(*)
    FROM fact f
    LEFT ANTI JOIN pondolens.dev_gold.dim_year y
        ON f.year_id = y.year_id

    UNION ALL

    SELECT
        7,
        'Region IDs missing or not regional',
        COUNT(*)
    FROM fact f
    LEFT ANTI JOIN pondolens.dev_gold.dim_regions r
        ON f.region_id = r.region_id
       AND LOWER(TRIM(r.geographic_level)) = 'region'

    UNION ALL

    SELECT
        8,
        'Indicator IDs missing from dim_indicator',
        COUNT(*)
    FROM fact f
    LEFT ANTI JOIN pondolens.dev_gold.dim_indicator d
        ON f.indicator_id = d.indicator_id

    UNION ALL

    SELECT
        9,
        'Indicator attributes do not match dimension',
        COUNT(*)
    FROM fact f
    LEFT ANTI JOIN pondolens.dev_gold.dim_indicator d
        ON f.indicator_id = d.indicator_id
       AND (f.indicator <=> d.indicator)
       AND (f.unit <=> d.unit)
       AND (f.service <=> d.service)
       AND (f.breakdown_value <=> d.breakdown_value)

    UNION ALL

    SELECT
        10,
        'Negative measurements',
        COUNT_IF(measurement < 0)
    FROM fact

    UNION ALL

    SELECT
        11,
        'Percentage measurements above 100',
        COUNT_IF(
            INSTR(unit, '%') > 0
            AND measurement > 100
        )
    FROM fact
)

SELECT
    c.dq_check,
    t.rows_inspected,
    c.affected_rows,
    ROUND(
        100.0 * c.affected_rows / NULLIF(t.rows_inspected, 0),
        2
    ) AS affected_percentage,
    CASE
        WHEN t.rows_inspected = 0 THEN 'NO DATA'
        WHEN c.affected_rows = 0 THEN 'PASS'
        WHEN 100.0 * c.affected_rows
             / NULLIF(t.rows_inspected, 0) <= 5 THEN 'REVIEW'
        ELSE 'FAIL'
    END AS status
FROM checks c
CROSS JOIN totals t
ORDER BY c.check_order;

## Results
- **Total:** 215 rows across 4 distinct indicator IDs.
- **GRDP:** 90 rows, covering 18 regions per year from 2021–2025.
- **Unemployment:** 87 rows, covering 17 regions per year in 2021–2023 and 18 regions per year in 2024–2025.
- **Water:** 34 rows, covering 17 regions in both 2022 and 2024.
- **Internet:** 4 rows, covering 4 regions in 2024.
- **Electricity:** No observations loaded.

## Data quality
- All 11 checks pass, with 0 affected rows.
- Checks cover missing fields, duplicate IDs and grain combinations, ID bounds, dimension references, attribute consistency, and measurement ranges.
- Status thresholds: **PASS = 0%**, **REVIEW = above 0% through 5%**, and **FAIL = above 5%**.
- Duplicate checks count all rows involved in duplicates.
- Checks assess loaded fact rows only; they do not detect observations excluded by joins or discarded during deduplication.
